# Titanic Survival Prediction — Custom Decision Tree and Random Forest

**Roll Number:** 25280091  
**Course:** AI500 — Machine Learning  
**Assignment:** Project 3 — Decision Trees & Random Forests  

---

### Overview

In this project, we build two models **from scratch** (without using scikit-learn classifiers):

1. **Part 1 — Custom Decision Tree**
2. **Part 2 — Custom Random Forest**
2. **Part 3 — SVM**

We train them on the **Titanic dataset** to predict passenger survival.  
We also analyze model accuracy and compare results.


**Import libraries**

In [1]:
# ==============================
# TO BE IMPLEMENTED
# ==============================
import pandas as pd
import numpy as np
# import are relvant libraries


### **Part 1 — Custom Decision Tree**

### Load and Prepare Titanic Dataset

In [2]:
# Load training data
data = pd.read_csv("titanic.csv")

print(data.head())
print(data.info())
print(data.isnull().sum())





   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

The Titanic dataset contains passenger information such as age, class, gender, fare, and survival status. Some columns contain missing values and categorical data that need to be handled properly.

---

## **Task: Implement Data Preprocessing Functions**

You are required to write Python functions (without using advanced preprocessing libraries like `sklearn.preprocessing`) to perform the following steps manually.

---

### *Step 1: Handle Missing Values*

Write functions to:

1. Replace missing values in the **Age** column with the **median age** of all passengers.  
2. Replace missing values in the **Embarked** column with the **most common embarkation port (mode)**.

---

### *Step 2: Drop Irrelevant Columns*

Remove the following columns that are not directly useful for survival prediction:

1. **Cabin** – too many missing values  
2. **Ticket** – non-numeric and not informative  
3. **Name** – mostly unique values  
4. **PassengerId** – identifier, not a feature

---

### **Step 3: Encode Categorical Variables**

Since machine learning models require numerical data, we need to convert categorical variables into numeric format manually.

#### Encode the `Sex` column:
- `male` → `0`  
- `female` → `1`

#### Encode the `Embarked` column:
- `C` → `0`  
- `Q` → `1`  
- `S` → `2`

This manual encoding ensures that categorical features can be interpreted correctly by machine learning algorithms, which typically work only with numerical inputs.


In [3]:
# ==============================
# TO BE IMPLEMENTED
# ==============================

# Handle missing values
def handle_missing_values(df):
	"""
	Fill Age missing values with median and Embarked with mode.
	Returns the modified DataFrame.
	"""
	# find the median value of the Age column
	median_age = df['Age'].median()

	# replace missing Age values with the median
	df['Age'] = df['Age'].fillna(median_age)

	# check if there are any missing values in Embarked column
	if df['Embarked'].isna().any():

		# find the most frequent value (mode) in Embarked
		mode_embarked = df['Embarked'].mode().iloc[0]

		# replace missing Embarked values with the mode
		df['Embarked'] = df['Embarked'].fillna(mode_embarked)

	# return the updated DataFrame
	return df



# Drop unused columns
def drop_irrelevant_columns(df):
	"""
	Drop Cabin, Ticket, Name, PassengerId as they are not predictive here.
	"""
	# drop columns that are not useful for prediction
	# errors='ignore' prevents error if any column is missing
	return df.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'], errors='ignore')



# Encode categorical features manually
def encode_categorical_features(df):
	"""
	Encode Sex and Embarked with fixed mappings.
	Sex: male->0, female->1
	Embarked: C->0, Q->1, S->2
	"""
	# mapping dictionary for Sex column
	sex_map = {'male': 0, 'female': 1}

	# convert Sex strings to numbers
	df['Sex'] = df['Sex'].map(sex_map).astype(int)

	# mapping dictionary for Embarked column
	embarked_map = {'C': 0, 'Q': 1, 'S': 2}

	# convert Embarked strings to numbers
	df['Embarked'] = df['Embarked'].map(embarked_map).astype(int)

	# return updated DataFrame
	return df



# Apply preprocessing steps

# first handle missing values
data = handle_missing_values(data)

# then drop the columns we don't need
data = drop_irrelevant_columns(data)

# then convert categorical text into numbers
data = encode_categorical_features(data)


Split data into train and test (Do NOT change following cell)

In [4]:
from sklearn.model_selection import train_test_split
# Split data into train and test
X = data.drop('Survived', axis=1)
y = data['Survived'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## **Task: Implement Core Functions for Decision Tree Splitting**

In this section, you will implement the key mathematical components that allow a **Decision Tree** to decide where to split data during training. You will write three main functions: `entropy`, `information_gain`, and `best_split`.

---

### **Step 1: Compute Entropy**
- The entropy function measures the impurity or uncertainty in a dataset.  
- Formula:  
  $$
  H(y) = -\sum_i p_i \log_2(p_i)
  $$
  where $p_i$ is the probability of each class label.
- Use NumPy operations to calculate probabilities.
- Add a small constant (e.g., `1e-9`) inside the log to avoid numerical errors.
- Test on simple arrays, e.g., `entropy([0, 0, 1, 1])`.

---

### **Step 2: Compute Information Gain**
- Information Gain (IG) quantifies how much entropy decreases after a split.
- Formula:  
  $$
  IG = H(\text{parent}) - \frac{n_{left}}{n}H(\text{left}) - \frac{n_{right}}{n}H(\text{right})
  $$
- Implement this using the `entropy()` function.
- The higher the IG, the better the feature/threshold for splitting.

---

### **Step 3: Find the Best Split**
- The `best_split()` function finds which feature and threshold produce the **maximum information gain**.
- Loop through all features (or a given subset) and:
  - For **numerical features**:
    - Try splitting at all unique threshold values.
    - Create boolean masks for left (`<= t`) and right (`> t`) splits.
  - For **categorical features**:
    - Split data based on equality (`== val`) vs. inequality (`!= val`).
- Skip invalid splits (e.g., when one side is empty).
- Return:
  - `best_feature`: the most informative feature,
  - `best_threshold`: the split value,
  - `best_gain`: the highest information gain achieved.

---

### **Expected Output**
- `entropy()` returns 0 when all samples belong to one class.
- `information_gain()` returns higher values for better splits.
- `best_split()` identifies the optimal feature and threshold for partitioning the data.

Implement the functions below step by step.


In [5]:
# ==========================
# TO BE IMPLEMENTED
# ==========================

def entropy(y):
    """
    Compute the entropy of a label array y.
    Formula: H(y) = -Σ p_i * log2(p_i)
    """
    y_arr = np.asarray(y)       # convert labels to numpy array
    if y_arr.size == 0:         # if no samples, entropy = 0
        return 0.0
    # Count occurrences of each class
    _, counts = np.unique(y_arr, return_counts=True)   # count how many of each label exist
    probs = counts / counts.sum()                      # convert counts to probabilities
    eps = 1e-9  # small constant for numerical stability
    h = -np.sum(probs * np.log2(probs + eps))          # formula of entropy
    # Clamp tiny negative due to eps/rounding
    return float(h) if h > 0 else 0.0                  # make sure no negative due to rounding


def information_gain(y, y_left, y_right):
    """
    Compute the information gain of a split.
    IG = H(parent) - (n_left/n)*H(left) - (n_right/n)*H(right)
    """
    y = np.asarray(y)              # convert to numpy array
    y_left = np.asarray(y_left)
    y_right = np.asarray(y_right)

    n = y.size                     # total samples
    if n == 0:                     # if no samples, IG = 0
        return 0.0

    n_left = y_left.size           # left child count
    n_right = y_right.size         # right child count
    # Invalid split if any side is empty
    if n_left == 0 or n_right == 0:
        return 0.0

    h_parent = entropy(y)          # entropy of parent
    h_left = entropy(y_left)       # entropy of left child
    h_right = entropy(y_right)     # entropy of right child

    w_left = n_left / n            # weight of left split
    w_right = n_right / n          # weight of right split

    gain = h_parent - (w_left * h_left + w_right * h_right)   # formula of IG
    return float(gain)


def best_split(X, y, feature_subset=None):
    """
    Find the best feature and threshold that maximize information gain.
    """
    y = np.asarray(y)                     # convert y to array
    if y.size == 0 or X.shape[0] == 0:    # no samples => no split
        return None, None, 0.0

    # if feature subset given (random forest) use that otherwise use all columns
    features = feature_subset if feature_subset is not None else list(X.columns)

    best_feature = None
    best_threshold = None
    best_gain = 0.0

    # loop each feature to test splits
    for feat in features:
        col = X[feat]                     # one column
        values = col.values               # get numpy values

        # Numeric feature
        if np.issubdtype(col.dtype, np.number):
            unique_vals = np.unique(values)   # find all possible thresholds
            for t in unique_vals:             # test each threshold
                left_mask = values <= t       # left side <= threshold
                right_mask = ~left_mask       # right side > threshold
                if not left_mask.any() or not right_mask.any():   # skip if empty
                    continue
                gain = information_gain(y, y[left_mask], y[right_mask])   # compute IG
                if gain > best_gain:          # update best split
                    best_gain = gain
                    best_feature = feat
                    best_threshold = t
        else:
            # Categorical feature: split on equality to each category
            unique_vals = pd.unique(values)   # all categories
            for v in unique_vals:
                left_mask = values == v       # category equals v
                right_mask = ~left_mask       # category not equal v
                if not left_mask.any() or not right_mask.any():
                    continue
                gain = information_gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feat
                    best_threshold = v

    # return feature name, threshold, IG value
    return best_feature, best_threshold, float(best_gain)


## **Task: Implement the Custom Decision Tree Classifier**

In this task, you will implement your own **Decision Tree** algorithm from scratch — without using libraries like `sklearn.tree`.  
This implementation will rely on the previously defined helper functions: `entropy`, `information_gain`, and `best_split`.

---

### **Class Overview**

You will create a `DecisionTree` class with the following methods:

---

### **1. `__init__()`**
- Initializes the tree’s hyperparameters:
  - `max_depth`: the maximum allowed depth of the tree.  
  - `min_samples_split`: minimum number of samples required to make a split.  
  - `feature_subsample`: number of randomly selected features (used in Random Forests).  
- The tree itself will be stored as a nested dictionary (`self.tree`).

---

### **2. `fit()`**
- Recursively builds the decision tree using **Information Gain** as the splitting criterion.
- **Base cases**:
  - If there are no samples left (`len(y) == 0`) → return `0`.
  - If all samples belong to the same class → return that class label.
  - If maximum depth is reached or too few samples remain → return the most common class (majority vote).
- **Recursive case**:
  1. Select the best feature and threshold using `best_split()`.
  2. Split the dataset into left and right subsets based on that threshold.
  3. Recursively call `fit()` on each subset to build subtrees.
- The final tree should be stored in `self.tree` as nested dictionaries with keys:
..., 'threshold': ..., 'gain': ..., 'left': ..., 'right': ...}


---

### **3. `predict_one()`**
- Predict the class for a **single data point**.
- Traverse the decision tree recursively:
- For numeric thresholds: use `<=` for the left branch, `>` for the right.
- For categorical values: use `==` for the left branch, `!=` for the right.
- Stop recursion when a leaf node (class label) is reached.

---

### **4. `predict()`**
- Predict the class for **an entire dataset** (`X`).
- Apply `predict_one()` to each row using a loop or list comprehension.
- Return predictions as a NumPy array.

---

### **Expected Behavior**
- The `fit()` method should build a decision tree automatically.
- The `predict()` method should output the correct class predictions for given input samples.
- You should be able to check accuracy by comparing predicted and actual labels.

---

### **Example Usage (after implementation)**

```python
tree = DecisionTree(max_depth=4)
tree.fit(X_train, y_train)
predictions = tree.predict(X_test)




In [6]:
# ==============================
# TO BE IMPLEMENTED
# ==============================

from collections import Counter

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2, feature_subsample=None):
        """
        Initialize the Decision Tree parameters.
        """
        self.max_depth = max_depth                   # how deep the tree is allowed
        self.min_samples_split = min_samples_split   # minimum samples needed to split
        # feature_subsample: if not None → only use these many random features (for RF)
        self.feature_subsample = feature_subsample
        self.tree = None                             # this will store the final learned tree

    def _majority_class(self, y):
        # return the most common class label in y
        return Counter(y).most_common(1)[0][0]

    def _build(self, X, y, depth):
        # if no samples → return dummy class 0
        if len(y) == 0:
            return 0
        # if all labels same → pure node → return that label
        if len(set(y)) == 1:
            return int(y[0])
        # stop if depth too big or samples too small → return majority label
        if depth >= self.max_depth or len(y) < self.min_samples_split:
            return int(self._majority_class(y))

        # if using random forest: choose subset of features at this node
        if self.feature_subsample is not None:
            k = min(self.feature_subsample, X.shape[1])         # take min of given k or total features
            feature_subset = list(np.random.choice(X.columns, size=k, replace=False))
        else:
            feature_subset = None                               # else use all features

        # find best feature and threshold from earlier function
        feat, thr, gain = best_split(X, y, feature_subset)

        # if no gain → no usefulness to split → return majority class
        if feat is None or gain <= 0:
            return int(self._majority_class(y))

        col = X[feat].values     # values of best feature

        # numeric feature → split by <= threshold
        if np.issubdtype(X[feat].dtype, np.number):
            left_mask = col <= thr
            right_mask = ~left_mask
        else:
            # categorical → split by == category
            left_mask = col == thr
            right_mask = ~left_mask

        X_left, y_left = X[left_mask], y[left_mask]     # left split
        X_right, y_right = X[right_mask], y[right_mask] # right split

        # if split invalid (one side empty) → majority class
        if len(y_left) == 0 or len(y_right) == 0:
            return int(self._majority_class(y))

        # recursively build children nodes
        left_node = self._build(X_left, y_left, depth + 1)
        right_node = self._build(X_right, y_right, depth + 1)

        # return node dictionary describing split
        return {
            'feature': feat,
            'threshold': thr,
            'gain': float(gain),
            'left': left_node,
            'right': right_node,
        }

    def fit(self, X, y, depth=0):
        """
        Recursively build the decision tree using information gain.
        """
        y_arr = np.asarray(y)                # convert to numpy array
        self.tree = self._build(X, y_arr, depth)  # construct root node
        return self

    def predict_one(self, x, node=None):
        """
        Predict class label for a single sample.
        """
        if node is None:
            node = self.tree                 # start at the root
        # if leaf → return class integer
        if not isinstance(node, dict):
            return int(node)
        feat = node['feature']               # which feature this node splits on
        thr = node['threshold']              # threshold/category
        val = x[feat]                        # value of this sample for that feature
        # numeric split
        if isinstance(thr, (int, float, np.integer, np.floating)):
            if val <= thr:
                return self.predict_one(x, node['left'])
            else:
                return self.predict_one(x, node['right'])
        else:
            # categorical split
            if val == thr:
                return self.predict_one(x, node['left'])
            else:
                return self.predict_one(x, node['right'])

    def predict(self, X):
        """
        Predict class labels for all rows in dataset X.
        """
        # predict for each row using predict_one
        preds = [self.predict_one(X.iloc[i]) for i in range(len(X))]
        return np.asarray(preds, dtype=int)


### **Train and Evaluate the Custom Decision Tree**

Do **not** change the following cell.  
It will be used to train, test, and evaluate your implemented `DecisionTree` class automatically.


In [7]:
tree = DecisionTree(max_depth=4)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

acc_tree = np.mean(y_pred_tree == y_test)
print(f"Custom Decision Tree Accuracy: {acc_tree:.4f}")

compare_tree = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_tree})
compare_tree.head(10)


Custom Decision Tree Accuracy: 0.7989


,Actual,Predicted
0,1,0
1,0,0
2,0,0
3,1,1
4,1,1
5,1,1
6,1,1
7,0,0
8,1,1
9,1,1


## **Part 2 — Custom Random Forest**

In this section, you will implement your own **Random Forest Classifier** from scratch — without using libraries like `sklearn.ensemble`.

A Random Forest combines multiple Decision Trees to form an ensemble model that improves prediction accuracy and reduces overfitting.

---

### **Instructions**

1. **Initialize Parameters (`__init__`)**
   - `n_estimators`: Number of decision trees to train.
   - `max_depth`: Maximum depth of each decision tree.
   - `min_samples_split`: Minimum number of samples required to split a node.
   - `feature_subsample_ratio`: Fraction of features to randomly select for each tree (e.g., 0.7 = 70% of features per tree).

   Store all trees in a list called `self.trees`.

---

2. **Train the Model (`fit`)**
   - For each tree:
     - Create a **bootstrap sample** by randomly selecting rows from the dataset **with replacement**.
     - Randomly choose a subset of features using the ratio `feature_subsample_ratio`.
     - Train a new instance of your `DecisionTree` class on the sampled data.
   - Append each trained tree to the list `self.trees`.

---

3. **Make Predictions (`predict`)**
   - Each trained decision tree makes predictions for all samples in `X`.
   - Combine predictions from all trees using **majority voting** (most common class).
   - Return the final predicted labels as a NumPy array.

---

### **Expected Behavior**
- The Random Forest should generally achieve **equal or better accuracy** than your single Decision Tree.
- The `fit()` and `predict()` methods should work seamlessly with your previously defined `DecisionTree` implementation.

---

### **Goal**
By completing this part, you will understand how ensemble learning improves model robustness through randomness and aggregation.


Implement Custom Random Forest Class

In [8]:
# ==============================
# TO BE IMPLEMENTED
# ==============================


from collections import Counter

class RandomForest:
    def __init__(self, n_estimators=10, max_depth=5, min_samples_split=2, feature_subsample_ratio=0.7):
        """
        Initialize the Random Forest parameters.
        """
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature_subsample_ratio = feature_subsample_ratio
        self.trees = []  # List to store all trained decision trees

    def fit(self, X, y):
        """
        Train multiple Decision Trees using bootstrapped samples and random feature subsets.
        """
        n_samples = X.shape[0]
        n_features = X.shape[1]
        
        # Calculate number of features to subsample for each tree
        n_features_subsample = max(1, int(n_features * self.feature_subsample_ratio))
        
        # Clear any previously trained trees
        self.trees = []
        
        # Train n_estimators trees
        for i in range(self.n_estimators):
            # Create bootstrap sample (with replacement)
            bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_bootstrap = X.iloc[bootstrap_indices].reset_index(drop=True)
            y_bootstrap = np.asarray(y)[bootstrap_indices]
            
            # Create a DecisionTree with feature subsampling
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                feature_subsample=n_features_subsample
            )
            
            # Train the tree on bootstrap sample
            tree.fit(X_bootstrap, y_bootstrap)
            
            # Store the trained tree
            self.trees.append(tree)

    def predict(self, X):
        """
        Predict class labels for all samples using majority voting from all trained trees.
        """
        if len(self.trees) == 0:
            raise ValueError("Random Forest has not been trained yet. Call fit() first.")
        
        n_samples = X.shape[0]
        all_predictions = []
        
        # Get predictions from all trees
        for tree in self.trees:
            predictions = tree.predict(X)
            all_predictions.append(predictions)
        
        # Convert to numpy array: shape (n_estimators, n_samples)
        all_predictions = np.array(all_predictions)
        
        # Majority voting: find the most common class for each sample
        final_predictions = []
        for i in range(n_samples):
            # Get predictions from all trees for sample i
            sample_predictions = all_predictions[:, i]
            # Use Counter to find the most common class (majority vote)
            most_common = Counter(sample_predictions).most_common(1)[0][0]
            final_predictions.append(most_common)
        
        return np.array(final_predictions, dtype=int)


### **Train and Evaluate the Custom Random Forest**

Do **not** change the following cell.  
It will be used to train, test, and evaluate your implemented `RandomForest` class automatically.


In [9]:
forest = RandomForest(n_estimators=10, max_depth=5)
forest.fit(X_train, y_train)
y_pred_forest = forest.predict(X_test)

acc_forest = np.mean(y_pred_forest == y_test)
print(f"Custom Random Forest Accuracy: {acc_forest:.4f}")

compare_forest = pd.DataFrame({
    'Actual': y_test,
    'Tree_Pred': y_pred_tree,
    'Forest_Pred': y_pred_forest
})
compare_forest.head(7)


Custom Random Forest Accuracy: 0.8101


,Actual,Tree_Pred,Forest_Pred
0,1,0,0
1,0,0,0
2,0,0,0
3,1,1,1
4,1,1,1
5,1,1,1
6,1,1,1


### **Comparison**

Do **not** change the following cell.  
It compares the performance of your **Custom Decision Tree** and **Custom Random Forest** implementations by printing their respective accuracies.


In [10]:
print(f"Decision Tree Accuracy: {acc_tree:.4f}")
print(f"Random Forest Accuracy: {acc_forest:.4f}")


Decision Tree Accuracy: 0.7989
Random Forest Accuracy: 0.8101


**Analytical Question:**  
Based on the accuracy results of the Decision Tree and Random Forest models, explain **why the Random Forest might perform better or worse** than a single Decision Tree.  
Discuss the roles of **ensemble learning**, **variance reduction**, and **overfitting** in your explanation.


**Answer:** 
Random Forest generally performs better than a Decision Tree because it is based on ensemble learning. Instead of using a single tree, it uses multiple decision trees, which helps reduce overfitting and variance. In terms of variance, a Decision Tree tends to overfit the training data by capturing noise, whereas a Random Forest averages the predictions from multiple decision trees each trained on a different subset of the data . thus generalizing the predictions better. Due to bootstrap sampling of the data and the combination of many trees, the risk of overfitting in a Random Forest is significantly reduced compared to a single Decision Tree.

---
## **Part 3 — Support Vector Machines (SVM) with Scikit-Learn**

In this final part, you will explore another powerful supervised learning algorithm: the **Support Vector Machine (SVM)**. Unlike Decision Trees, which make splits by partitioning data, an SVM's goal is to find the optimal hyperplane or decision boundary that best separates the classes.

You will use `scikit-learn`'s implementation to train an SVM on the same preprocessed Titanic data and compare its performance to your from-scratch models.

### **Task: Train and Evaluate SVM Classifiers**

You will use `scikit-learn`'s `SVC` (Support Vector Classifier). We will experiment with two key hyperparameters: the `kernel` and the regularization parameter `C`.

#### **Understanding SVM Kernels**
The **kernel** allows the SVM to find complex, non-linear decision boundaries. We will use two types:
* **`linear` kernel:** This attempts to separate the data with a single straight line (or a flat plane in higher dimensions). It's fast and works well if the data is linearly separable. Think of it as using a ruler to divide two groups of points.
* **`rbf` kernel (Radial Basis Function):** This is the default and more flexible kernel. It can create complex, curved decision boundaries by mapping the data to a higher-dimensional space. Think of it as using a flexible wire to enclose different groups of points.

#### **Understanding the Regularization Parameter `C`**
The **`C` parameter** controls the trade-off between achieving a low training error and a low testing error.
* A **low `C`** makes the decision boundary smoother and the margin wider. The model is more tolerant of misclassifying a few training points, which can lead to better **generalization** (less overfitting).
* A **high `C`** aims to classify every training example correctly, which can lead to a more complex boundary and a narrower margin. This might **overfit** the training data.

**Instructions:**
1.  Use the same `X_train`, `y_train`, `X_test`, and `y_test` data from Part 1.
2.  Train one `SVC` model with a `linear` kernel as a baseline.
3.  Train three separate `SVC` models with an `rbf` kernel, using `C` values of `0.1`, `1`, and `10`.
4.  For each of the four models, calculate and print its accuracy on the test set.

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Train and evaluate an SVM with a linear kernel
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
acc_svm_linear = accuracy_score(y_test, y_pred_linear)
print(f"Linear SVM Accuracy: {acc_svm_linear:.4f}")

# 3. Train and evaluate SVMs with an RBF kernel and different C values
rbf_accuracies = {}
for C_value in [0.1, 1, 10]:
    svm_rbf = SVC(kernel='rbf', C=C_value, gamma='scale')
    svm_rbf.fit(X_train_scaled, y_train)
    y_pred_rbf = svm_rbf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_rbf)
    rbf_accuracies[C_value] = acc
    print(f"RBF SVM Accuracy (C={C_value}): {acc:.4f}")

Linear SVM Accuracy: 0.7821
RBF SVM Accuracy (C=0.1): 0.8101
RBF SVM Accuracy (C=1): 0.8156
RBF SVM Accuracy (C=10): 0.7989


### **Final Comparison and Analysis**

Now, let's compare the performance of all the models you've worked with.

In [12]:
#Replace the accuracy variables with your defined variables for SVM accuracies

print(f"Custom Decision Tree Accuracy: {acc_tree:.4f}")
print(f"Custom Random Forest Accuracy: {acc_forest:.4f}")
print(f"Scikit-learn Linear SVM Accuracy: {acc_svm_linear:.4f}")
for c, acc in rbf_accuracies.items():
    print(f"Scikit-learn RBF SVM Accuracy (C={c}): {acc:.4f}")

Custom Decision Tree Accuracy: 0.7989
Custom Random Forest Accuracy: 0.8101
Scikit-learn Linear SVM Accuracy: 0.7821
Scikit-learn RBF SVM Accuracy (C=0.1): 0.8101
Scikit-learn RBF SVM Accuracy (C=1): 0.8156
Scikit-learn RBF SVM Accuracy (C=10): 0.7989


**Analytical Question:**
1.  Looking at the RBF SVM results, how did changing the `C` parameter affect the model's accuracy? Based on the explanation above, what does your result suggest about the trade-off between a wide margin and fitting the training data for this problem?
2.  Compared to your Random Forest, what is one advantage and one disadvantage of using an SVM? (Consider factors like interpretability, training time, and prediction performance).

**Answer:**

1) At C = 0.1 the model has a wider margin , tolerates some misclassifications and generalizes well but when increased from 0.1 to 1 the model accuracy improved slightly better reducing bias. However when increased to 10 the accuracy decreased due to overfitting it started fitting noise rather than learning the generic pattern. 

So here is the conclusion according to the results ,

- A small C gives wider marging and better generalization

- A large C gives a narrower margin and risks overfitting.

2)  Advantage: 

Svm with kernel functions RBF can capture complex , non-linear relationships even when the dataset is small or has many features. SVM performs well in such cases where random forest might struggle to find the splits.

    Disadvantage: 

SVM is difficult to understand how individual feature influences the prediction , unlike random forest which provides feature importance measures thus SVMs are less interpretable. Moreover , partiparticularly when using with RBF kernels it can be computationally expensive and is slower to train on large datasets compared to Random forests.